In [25]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import  MultinomialNB, BernoulliNB, ComplementNB
import joblib

In [26]:
str(LinearSVC(), )

'LinearSVC()'

In [27]:
df = pd.read_csv("data/finished_products.csv")

In [28]:
df.category_label = df.category_label.astype("category")

In [29]:
df

,product_title,category_label
0,apple iphone 8 plus 64gb silver,mobile_phones
1,apple iphone 8 plus 64 gb spacegrau,mobile_phones
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,mobile_phones
3,apple iphone 8 plus 64gb space grey,mobile_phones
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,mobile_phones
...,...,...
30881,smeg fab28 60cm retro style right hand hinge f...,fridges
30882,smeg fab28 60cm retro style left hand hinge fr...,fridges
30883,smeg fab28 60cm retro style left hand hinge fr...,fridges
30884,candy 60cm built under larder fridge cru160nek,fridges


In [30]:
X = df[["product_title"]]
y = df["category_label"]

In [31]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)

In [32]:
X_train

,product_title
26632,smeg fc400x2pe fridge freezer
6400,lg 42lb5610 full hd led television
15376,zanussi zdf22002xa full size dishwasher stainl...
20627,beko wtb820e1w 8 kg 1200 spin washing machine ...
5145,samsung ue55nu7300 55 curved ultra hd certifie...
...,...
18979,hoover dxoc510c3/dxoc510c3
13177,panasonic nn st479sbpq family solo auto sensor...
16096,beko dis15010 slimline integrated dishwasher
29022,smeg fab5rrd 40cm red small 50 s style right h...


In [33]:
def train_model(model):
    preprocessor = ColumnTransformer(
        transformers= [
            ("title", TfidfVectorizer(), "product_title"),
        ]
    )

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", model)
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    c_report = classification_report(y_test, y_pred, output_dict=True)

    metrics = {
    "model": str(model)[:-2],
    "model_weights" : pipeline,
    "average_score": round(c_report["accuracy"], 3),
    "precision": round(c_report["macro avg"]["precision"], 3),
    "f1-score": round(c_report["macro avg"]["f1-score"], 3),
    "recall": round(c_report["macro avg"]["recall"], 3),
    }
    return metrics



In [12]:
# 'macro avg': {'precision': 0.9740398156374849,
#   'recall': 0.9649694714140735,
#   'f1-score': 0.9693360117701766,
#   'support': 6178.0},

In [34]:
results = []
for i in [RandomForestClassifier(), LinearSVC(), SVC(), MultinomialNB(), BernoulliNB(), ComplementNB()]:
    print(i)
    in_rez= train_model(i)
    print(in_rez)
    results.append(in_rez)

RandomForestClassifier()
{'model': 'RandomForestClassifier', 'model_weights': Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('title', TfidfVectorizer(),
                                                  'product_title')])),
                ('classifier', RandomForestClassifier())]), 'average_score': 0.964, 'precision': 0.972, 'f1-score': 0.965, 'recall': 0.959}
LinearSVC()
{'model': 'LinearSVC', 'model_weights': Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('title', TfidfVectorizer(),
                                                  'product_title')])),
                ('classifier', LinearSVC())]), 'average_score': 0.97, 'precision': 0.978, 'f1-score': 0.971, 'recall': 0.965}
SVC()
{'model': 'SVC', 'model_weights': Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('title', TfidfVectorizer(),
                                                  'product_title')])),
               

In [14]:
model = max(results, key=lambda d: d["f1-score"])["model_weights"]


In [15]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](9,)","['cpus','digital_cameras','dishwashers',...,'mobile_phones','tvs', 'washing_machines']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['product_title','merchant_id','product_family','special_name']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('title', ...), ('merchant', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedco

In [35]:
joblib.dump(model, "models/best_model.joblib")

['models/best_model.joblib']

In [36]:
df = pd.read_csv("data/finished_products.csv").head(5)

In [37]:
df

,product_title,category_label
0,apple iphone 8 plus 64gb silver,mobile_phones
1,apple iphone 8 plus 64 gb spacegrau,mobile_phones
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,mobile_phones
3,apple iphone 8 plus 64gb space grey,mobile_phones
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,mobile_phones


In [39]:
X = df[["product_title"]]
y = df["category_label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)

In [40]:
X_train

,product_title
4,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...
2,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...
1,apple iphone 8 plus 64 gb spacegrau
0,apple iphone 8 plus 64gb silver


In [41]:
def retrive_model(path = "models/best_model.joblib"):
    return joblib.load(path)


In [42]:
X_test

,product_title
3,apple iphone 8 plus 64gb space grey


In [43]:
model_pipeline = retrive_model()
model_pipeline.predict(X_test[["product_title"]])

ValueError: columns are missing: {'special_name', 'merchant_id', 'product_family'}